# SWaT preprocessing check

This notebook runs `prepare_swat` from `src/data_loaders/preprocessing.py` and shows what it produces. It saves nothing.

Steps in the module: drop the first 6 hours of the normal file, drop 8 sensors (6 that never change, plus P201 and P403), split the normal data in time into train (85%) and validation (15%), and min-max scale every sensor using the training part only. Windows of 60 seconds are cut afterwards: stride 10 for training, stride 1 for validation and test.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data_loaders.preprocessing import EVAL_STRIDE, make_windows, prepare_swat

d = prepare_swat(ROOT / "data" / "raw" / "swat")
feats = d["features"]
print("features:", len(feats))
print("dropped (constant):", d["dropped_constant"], "| excluded:", d["excluded"])
for k in ("train", "val", "attack"):
    print(f"{k:<6}", d[k].shape, d[f"{k}_timestamps"][0], "->", d[f"{k}_timestamps"][-1])
print("train windows:", make_windows(d["train"]).shape)
print("test windows: ", make_windows(d["attack"], stride=EVAL_STRIDE).shape)
print("attack seconds:", int(d["attack_labels"].sum()), "of", len(d["attack_labels"]))

## One window

A window is 60 consecutive seconds of all 43 sensors, one row per second. Values are scaled so the training data runs from 0 to 1.

In [ ]:
pd.DataFrame(make_windows(d["train"])[0], columns=feats).head(10)

## Three sensors over the attack file

One point per second. Red points are seconds labelled attack.
- LIT301 and FIT501 leave the training range (0 to 1) only during attacks.
- P102 is a spare pump that never moves in the normal file. It is kept because it switches on during some attacks.

Some analyser sensors (AIT201, AIT203, AIT402, AIT502) drift slowly across the attack days, so they sit outside the training range even in seconds labelled Normal. Their scaled values are not plotted here.

The attack file labels attack #27 (P-302 and LIT-401, start 31 Dec 01:17) as attack until 11:15, about 10 hours. The attack list gives it about 28 minutes. This one block is 35,900 of the 54,621 attack seconds (66%), so per-second scores mostly reflect it. The labels are used as released; why the label runs longer is not established. Because of this, results should also report per-attack detection (eTaPR), not only point-wise F1.

In [ ]:
t = d["attack_timestamps"]
lab = d["attack_labels"]
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
for ax, name in zip(axes, ["LIT301", "FIT501", "P102"]):
    y = d["attack"][:, feats.index(name)]
    ax.plot(t, y, lw=0.6)
    ax.scatter(t[lab == 1], y[lab == 1], s=2, c="red")
    ax.set_ylabel(name)
axes[0].set_title("Scaled sensors over the attack file (0 to 1 = training range)")
plt.tight_layout()
plt.show()